# Modélisation du remboursement anticipé par hasard discret

Ce notebook présente une approche de modélisation du remboursement anticipé (RA) de crédits immobiliers fondée sur l'analyse de survie en temps discret (*discrete-time hazard model*).

L'idée centrale est d'estimer, pour chaque mois de vie d'un contrat, la probabilité conditionnelle de remboursement anticipé sachant que le contrat a survécu jusqu'à ce mois. Cette probabilité est la **fonction de hasard discrète** $h(t)$.

Le notebook est structuré en cinq étapes :

1. Préparation des données
2. Construction du panel personne-période et estimation du hasard brut
3. Modélisation logistique du hasard avec covariables
4. Prédiction, calibration et évaluation sur le jeu de test
5. Backtesting temporel sur fenêtres disjointes

**Choix structurant : définition unique de l'événement**

Dans ce notebook, l'événement de remboursement anticipé est défini uniformément par `EVENT = (RA > 0)` (montant de RA strictement positif) sur l'ensemble des étapes. La variable `FLAG_ER` n'est utilisée que pour les statistiques descriptives initiales. Ce choix doit être documenté et validé métier : si `FLAG_ER` et `(RA > 0)` divergent significativement, cela mérite une investigation spécifique.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import patsy
from patsy import bs
from scipy.special import logit, expit
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Constantes globales
ID_COL   = "ID"
DATE_COL = "Date_origination"
MAX_AGE  = 60  # horizon maximal d'analyse (mois)

print("Bibliothèques chargées")

## Etape 1 : Préparation des données

Les données sources sont au format transversal (une ligne par contrat, observé à un instant donné). On effectue les traitements préliminaires suivants :

- **Filtre sur la maturité** (`B_MAT <= 170`) : exclusion des contrats à maturité initiale aberrante.
- **Nettoyage** : suppression des colonnes dégénérées (`MREVAU`, `MREVNU`).
- **Recodage de `DARRET`** : format `YYYYMM` converti en `YYYY-MM-01`.
- **Harmonisation de l'EAD** : renommage depuis `E_EAD` si nécessaire. Les lignes sans EAD sont supprimées.
- **Traitement du RA** : valeurs manquantes imputées à 0, valeurs négatives exclues.
- **Imputation des valeurs manquantes** : `Tx` imputé par médiane intra-produit (médiane globale en fallback) ; `AGE_CLI` par médiane globale. La variable `MREVTOT` (revenu total) est conservée et imputée par médiane globale — elle est incluse dans le modèle comme covariable candidate.
- **Exclusion du produit 'AR'** : produit de nature différente, hors périmètre de l'analyse.
- **Filtre `Tx > 0`** : contrats sans taux sont probablement mal renseignés.
- **Calcul de l'âge du prêt** : `AGE_PRET = B_MAT - B_RESMAT` (mois écoulés depuis origination), axe temporel de l'analyse de survie.

> **Limite de la structure transversale** : on n'observe pas le processus longitudinal complet de chaque contrat. L'estimation du hasard repose sur l'hypothèse que la distribution des âges observés dans le portefeuille est représentative du processus de survie individuel. Cette hypothèse est discutable en présence d'effets millésime forts.

In [ ]:
df = pd.read_csv("Export_ER_Agrege.csv", sep=";")

# 1) Filtre maturité
df = df[(df["B_MAT"] <= 170) | (df["B_MAT"].isna())].copy()

# 2) Suppression colonnes inutiles
cols_to_drop = ["Unnamed: 19", "MREVAU", "MREVNU"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# 3) DARRET : YYYYMM -> YYYY-MM-01
df["DARRET"] = pd.to_datetime(df["DARRET"].astype(str) + "01",
                               format="%Y%m%d", errors="coerce")

# 4) Harmonisation EAD
if "E_EAD" in df.columns:
    df["EAD"] = df["E_EAD"]
    df = df.drop(columns=["E_EAD", "E_ONB"], errors="ignore")
df = df.dropna(subset=["EAD"])

# 5) RA : NA -> 0, filtre >= 0
df["RA"] = df["RA"].fillna(0)
df = df[df["RA"] >= 0]

# 6) Typage catégories
for c in ["CLASSACT", "CSP", "produit"]:
    if c in df.columns:
        df[c] = df[c].astype("category")

# 7) Imputation valeurs manquantes
if "Tx" in df.columns:
    if "produit" in df.columns:
        df["Tx"] = df.groupby("produit", observed=True)["Tx"].transform(
            lambda x: x.fillna(x.median())
        )
    df["Tx"] = df["Tx"].fillna(df["Tx"].median())

if "CSP" in df.columns:
    df["CSP"] = df["CSP"].fillna("Inconnu")

if "AGE_CLI" in df.columns:
    df["AGE_CLI"] = df["AGE_CLI"].fillna(df["AGE_CLI"].median())

# MREVTOT : conservé et imputé (covariable candidate au modèle)
if "MREVTOT" in df.columns:
    df["MREVTOT"] = df["MREVTOT"].fillna(df["MREVTOT"].median())

# 8) Exclusion produit 'AR'
if "produit" in df.columns:
    df = df[df["produit"] != "AR"]

# 9) Filtre Tx > 0
if "Tx" in df.columns:
    df = df[df["Tx"] > 0]

# 10) Calcul AGE_PRET
df = df.dropna(subset=["B_MAT", "B_RESMAT"])
df["AGE_PRET"]    = (df["B_MAT"] - df["B_RESMAT"]).astype(int)
df = df[(df["AGE_PRET"] >= 0) & (df["AGE_PRET"] <= df["B_MAT"])]
df["HORIZON_RES"] = df["B_RESMAT"].astype(int)

# 11) Date origination
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# 12) Définition unique de l'événement : RA montant > 0
df["EVENT"] = (df["RA"] > 0).astype(int)

# Nettoyage final
df = df.dropna(subset=[ID_COL, "AGE_PRET", "EVENT"]).copy()

print(f"Contrats après nettoyage : {df[ID_COL].nunique():,}")
print(f"Lignes totales            : {len(df):,}")
print(f"Taux d'événement global   : {df['EVENT'].mean():.4f}")
print(f"Ages observés             : {df['AGE_PRET'].min()} à {df['AGE_PRET'].max()} mois")

### Vérification de cohérence : FLAG_ER vs EVENT

On compare ici les deux définitions possibles de l'événement pour identifier d'éventuelles divergences. Un écart significatif doit faire l'objet d'une investigation métier avant tout choix définitif.

In [ ]:
if "FLAG_ER" in df.columns:
    df["FLAG_ER"] = df["FLAG_ER"].fillna(0).astype(int)
    n_total      = len(df)
    n_agree      = (df["EVENT"] == df["FLAG_ER"]).sum()
    n_ra_only    = ((df["EVENT"] == 1) & (df["FLAG_ER"] == 0)).sum()
    n_flag_only  = ((df["EVENT"] == 0) & (df["FLAG_ER"] == 1)).sum()

    print(f"Cohérence EVENT == FLAG_ER : {n_agree / n_total:.2%} des contrats")
    print(f"RA > 0 mais FLAG_ER = 0   : {n_ra_only:,} ({n_ra_only / n_total:.2%})")
    print(f"FLAG_ER = 1 mais RA = 0   : {n_flag_only:,} ({n_flag_only / n_total:.2%})")
    print("\n=> On retient EVENT = (RA > 0) comme définition opérationnelle.")
else:
    print("FLAG_ER absent du dataset — EVENT = (RA > 0) utilisé par défaut.")

## Etape 2 : Construction du panel personne-période et estimation du hasard brut

### Définition du hasard discret

$$h(t) = P(T = t \mid T \geq t)$$

L'estimateur empirique sur les données transversales est :

$$\hat{h}(t) = \frac{\text{Nb de RA au mois } t}{\text{Nb de contrats à risque au mois } t}$$

### Construction du panel

Chaque contrat génère autant de lignes que de mois de vie observés ($t = 0, \ldots, T_i$). L'indicateur `y = 1` est positionné au dernier mois uniquement si le contrat a réalisé un RA avant l'horizon `MAX_AGE`. Les contrats avec $T_i >$ `MAX_AGE` sont tronqués et traités comme censurés à droite.

In [ ]:
df_base = df.copy()

df_base["T_STOP_TRUE"]  = df_base["AGE_PRET"]
df_base["T_STOP_PANEL"] = df_base["AGE_PRET"].clip(lower=0, upper=MAX_AGE)

# Construction du panel personne-période
panel = (
    df_base[[ID_COL, "EVENT", "T_STOP_TRUE", "T_STOP_PANEL"]]
    .dropna(subset=[ID_COL, "T_STOP_PANEL"])
    .assign(t=lambda d: d["T_STOP_PANEL"].map(lambda T: list(range(int(T) + 1))))
    .explode("t", ignore_index=True)
)
panel["t"] = panel["t"].astype(int)

# y = 1 seulement si event ET event survient avant MAX_AGE ET c'est le dernier mois
panel["y"] = (
    (panel["EVENT"] == 1)
    & (panel["T_STOP_TRUE"] <= MAX_AGE)
    & (panel["t"] == panel["T_STOP_TRUE"])
).astype(int)

# Vérification de cohérence
events_panel    = panel["y"].sum()
events_base     = ((df_base["EVENT"] == 1) & (df_base["T_STOP_TRUE"] <= MAX_AGE)).sum()
assert events_panel == events_base, "Incohérence entre panel et base : vérifier la construction."

print(f"Lignes panel             : {len(panel):,}")
print(f"Evénements dans le panel : {events_panel:,}")
print(f"Check cohérence          : OK ({events_panel} == {events_base})")

In [ ]:
# Hasard brut par âge
hazard_df = (
    panel.groupby("t", as_index=False)
    .agg(Nb_at_risk=("y", "size"), Nb_RA=("y", "sum"))
)
hazard_df["Hazard_brut"] = hazard_df["Nb_RA"] / hazard_df["Nb_at_risk"]

print(hazard_df.tail(10).to_string(index=False))

### Courbe de survie et taux d'extinction bruts

A partir du hasard brut, on dérive la fonction de survie et les courbes d'extinction :

$$S(t) = \prod_{u=0}^{t}(1 - h(u))$$

$$\text{ER\_marginal}(t) = S(t-1) \cdot h(t) \qquad \text{ER\_cumulée}(t) = 1 - S(t)$$

In [ ]:
h = hazard_df.sort_values("t").copy()
h["Hazard_brut"] = h["Hazard_brut"].clip(0, 1)
h["S_t"]         = (1 - h["Hazard_brut"]).cumprod()
h["S_prev"]      = h["S_t"].shift(1).fillna(1.0)
h["ER_marginal"] = h["S_prev"] * h["Hazard_brut"]
h["ER_cumulee"]  = 1 - h["S_t"]

er_df = h[["t", "Hazard_brut", "S_t", "ER_marginal", "ER_cumulee"]].copy()

print(er_df.head(10).to_string(index=False))
print(f"\nER cumulée à t={MAX_AGE} : {er_df['ER_cumulee'].iloc[-1]:.4f}")

## Etape 3 : Modélisation logistique du hasard avec covariables

On estime le modèle de hasard discret suivant :

$$\text{logit}\left(h_i(t)\right) = f(t) + \beta_1 \tilde{T}x_i + \beta_2 \widetilde{\text{AGE\_CLI}}_i + \beta_3 \widetilde{\text{MREVTOT}}_i + \sum_k \gamma_k \mathbf{1}_{\text{produit}_i = k}$$

où $f(t)$ est une spline cubique (6 degrés de liberté) modélisant la dépendance temporelle non-paramétrique, et les variables continues sont centrées-réduites sur le train set.

### Split temporel

L'évaluation est réalisée par un **split temporel strict** : les 80% de contrats les plus anciens constituent le train, les 20% les plus récents le test. Ce choix reflète l'utilisation en production (prédiction sur des contrats futurs) et évite le biais d'optimisme d'un split aléatoire.

### Sélection des covariables

Trois covariables continues sont incluses : le taux contractuel (`Tx`), l'âge du client (`AGE_CLI`), et le revenu total mensuel (`MREVTOT`). La variable `MREVTOT` est incluse dans cette version corrigée ; son retrait éventuel doit être justifié par un critère explicite (taux de NA résiduel, contribution marginale au AIC/BIC, stabilité entre folds).

In [ ]:
base = df_base.copy()

# Split temporel 80/20 sur la date d'origination
cut = base[DATE_COL].quantile(0.80)
train_ids = set(base.loc[base[DATE_COL] <= cut, ID_COL])
test_ids  = set(base.loc[base[DATE_COL] > cut,  ID_COL])

print(f"Date de coupure train/test : {cut.date()}")
print(f"Contrats train : {len(train_ids):,}  |  Contrats test : {len(test_ids):,}")

# Covariables à merger depuis la base contrat
# On inclut MREVTOT si disponible
cov_cols = [ID_COL, "Tx", "AGE_CLI", "produit"]
if "MREVTOT" in base.columns:
    cov_cols.append("MREVTOT")

panel_cov = panel.merge(base[cov_cols], on=ID_COL, how="left")

# Regroupement des modalités produit peu fréquentes (top 15 global)
top_prod = panel_cov["produit"].value_counts().head(15).index
panel_cov["produit2"] = panel_cov["produit"].where(panel_cov["produit"].isin(top_prod), "Other")

# Split
needed = ["y", "t", "Tx", "AGE_CLI", "produit2"]
if "MREVTOT" in panel_cov.columns:
    needed.append("MREVTOT")

train = panel_cov[panel_cov[ID_COL].isin(train_ids)].dropna(subset=needed).copy()
test  = panel_cov[panel_cov[ID_COL].isin(test_ids)].dropna(subset=needed).copy()

print(f"\nLignes train : {len(train):,}  |  Events train : {int(train['y'].sum()):,}")
print(f"Lignes test  : {len(test):,}   |  Events test  : {int(test['y'].sum()):,}")

In [ ]:
# Standardisation sur le train set uniquement (pas de leakage)
def standardize(train_col, test_col):
    mu, sigma = train_col.mean(), train_col.std(ddof=0)
    sigma = sigma if sigma > 0 else 1.0
    return (train_col - mu) / sigma, (test_col - mu) / sigma, mu, sigma

train["Tx_z"], test["Tx_z"], tx_mean, tx_std = standardize(train["Tx"], test["Tx"])
train["AGE_CLI_z"], test["AGE_CLI_z"], age_mean, age_std = standardize(train["AGE_CLI"], test["AGE_CLI"])

has_mrevtot = "MREVTOT" in train.columns
if has_mrevtot:
    train["MREVTOT_z"], test["MREVTOT_z"], rev_mean, rev_std = standardize(train["MREVTOT"], test["MREVTOT"])

# Formule du modèle
# Note : bs(t) remplace les termes polynomiaux t, t^2, t^3 utilisés dans
# la version initiale. La spline est plus flexible et évite les artefacts
# d'extrapolation aux bords.
if has_mrevtot:
    formula = "y ~ bs(t, df=6, degree=3) + Tx_z + AGE_CLI_z + MREVTOT_z + C(produit2)"
else:
    formula = "y ~ bs(t, df=6, degree=3) + Tx_z + AGE_CLI_z + C(produit2)"

print(f"Formule : {formula}")

y_tr, X_tr = patsy.dmatrices(formula, data=train, return_type="dataframe")
glm = sm.GLM(y_tr, X_tr, family=sm.families.Binomial())
res = glm.fit()

print(res.summary().tables[1])

### Interprétation des coefficients

- **Baseline `bs(t, df=6, degree=3)`** : les coefficients croissants puis légèrement décroissants traduisent une forme en cloche du hasard en fonction de l'âge. Le risque de RA augmente durant les premières années du prêt (pic autour de 4-5 ans), puis se stabilise. Ce profil est typique des crédits immobiliers à taux fixe en France.

- **`Tx_z`** : le coefficient attendu est négatif — un taux contractuel plus élevé (relativement à la moyenne du portefeuille) est associé à une probabilité de RA plus forte. Les emprunteurs à taux élevé ont davantage intérêt à renégocier lors d'une baisse des taux de marché.

- **`AGE_CLI_z`** : l'effet de l'âge du client est attendu négatif mais d'amplitude modérée.

- **`MREVTOT_z`** : un revenu plus élevé peut être associé à une capacité de remboursement anticipé accrue, mais l'effet net dépend des interactions avec le taux et le type de produit. A interpréter avec prudence si le taux de valeurs manquantes était élevé avant imputation.

## Etape 4 : Prédiction, calibration et évaluation

### Construction de la courbe portefeuille prédite

Pour chaque contrat $i$ du test set, on calcule le hasard prédit $\hat{h}_i(t)$ puis la survie individuelle :

$$S_i(t) = \prod_{u=0}^{t}\left(1 - \hat{h}_i(u)\right)$$

La courbe portefeuille est la moyenne des survies individuelles : $\bar{S}(t) = \frac{1}{N}\sum_i S_i(t)$.

### Calibration par décalage logit

Si la courbe prédite présente un biais systématique en niveau par rapport à la courbe observée, on applique une calibration par décalage sur l'échelle logit : on cherche $\delta^*$ tel que l'ER finale prédite après décalage égale l'ER observée sur le test.

$$\hat{h}_i^{\text{cal}}(t) = \sigma\left(\sigma^{-1}\left(\hat{h}_i(t)\right) + \delta^*\right)$$

Cette approche préserve la discrimination du modèle (classement relatif des contrats) tout en corrigeant le niveau moyen.

> **Important** : la cible de calibration est calculée sur les **contrats du test set uniquement**, à partir de la variable `EVENT` définie avant toute modélisation. On évite ainsi toute contamination train/test dans la procédure de calibration.

In [ ]:
# 1) Hasard prédit sur le test
_, X_te = patsy.dmatrices(formula, data=test.assign(y=0), return_type="dataframe")
p0 = res.predict(X_te).clip(1e-9, 1 - 1e-9)

# 2) Courbe portefeuille prédite (non calibrée)
def compute_portfolio_curve(ids, times, hazards):
    """Calcule la courbe S(t) portefeuille à partir des hasards individuels."""
    tmp = pd.DataFrame({ID_COL: ids, "t": times, "p": hazards}).sort_values([ID_COL, "t"])
    tmp["S_i_t"] = tmp.groupby(ID_COL)["p"].transform(lambda s: (1 - s).cumprod())
    S_port = tmp.groupby("t", as_index=False).agg(S_t=("S_i_t", "mean")).sort_values("t")
    S_port["ER_cumulee"]  = 1 - S_port["S_t"]
    S_port["ER_marginal"] = S_port["S_t"].shift(1).fillna(1.0) - S_port["S_t"]
    return S_port

curve_pred = compute_portfolio_curve(test[ID_COL].values, test["t"].values, p0)
print(f"ER cumulée prédite (t={MAX_AGE}) : {curve_pred['ER_cumulee'].iloc[-1]:.4f}")

# 3) Courbe portefeuille observée (test)
haz_obs = test.groupby("t", as_index=False).agg(Nb_at_risk=("y","size"), Nb_RA=("y","sum"))
haz_obs["h_obs"]        = haz_obs["Nb_RA"] / haz_obs["Nb_at_risk"]
haz_obs["S_t_obs"]      = (1 - haz_obs["h_obs"]).cumprod()
haz_obs["ER_cumulee"]   = 1 - haz_obs["S_t_obs"]
haz_obs["ER_marginal"]  = haz_obs["S_t_obs"].shift(1).fillna(1.0) - haz_obs["S_t_obs"]

print(f"ER cumulée observée (t={MAX_AGE}) : {haz_obs['ER_cumulee'].iloc[-1]:.4f}")

# 4) Métriques avant calibration
cmp = haz_obs[["t","ER_marginal"]].rename(columns={"ER_marginal":"obs"}).merge(
      curve_pred[["t","ER_marginal"]].rename(columns={"ER_marginal":"pred"}), on="t")
mae_before  = (cmp["obs"] - cmp["pred"]).abs().mean()
rmse_before = np.sqrt(((cmp["obs"] - cmp["pred"])**2).mean())
print(f"\nAvant calibration — MAE : {mae_before:.6f}  |  RMSE : {rmse_before:.6f}")

In [ ]:
# Cible de calibration : ER finale observée sur les contrats du test
# calculée uniquement à partir de EVENT (défini avant modélisation, pas de leakage)
test_contracts = df_base[df_base[ID_COL].isin(test[ID_COL].unique())]
target_er = test_contracts["EVENT"].mean()
print(f"Cible de calibration (ER observée sur les contrats test) : {target_er:.6f}")
print(f"Nb contrats test : {test_contracts.shape[0]:,}")

def ER_final_with_delta(delta, ids, times, p_base):
    p_shifted = expit(logit(p_base) + delta)
    curve = compute_portfolio_curve(ids, times, p_shifted)
    return curve["ER_cumulee"].iloc[-1]

# Recherche binaire du delta optimal
lo, hi = -10.0, 10.0
for _ in range(40):
    mid = (lo + hi) / 2
    if ER_final_with_delta(mid, test[ID_COL].values, test["t"].values, p0) < target_er:
        lo = mid
    else:
        hi = mid

delta_star = (lo + hi) / 2
er_check = ER_final_with_delta(delta_star, test[ID_COL].values, test["t"].values, p0)
print(f"\ndelta* = {delta_star:.6f}")
print(f"ER finale après calibration : {er_check:.6f}  (cible : {target_er:.6f})")

In [ ]:
# Courbe calibrée finale
p_cal = expit(logit(p0) + delta_star)
curve_cal = compute_portfolio_curve(test[ID_COL].values, test["t"].values, p_cal)
curve_cal = curve_cal.rename(columns={"ER_cumulee": "ER_cumulee_cal", "ER_marginal": "ER_marginal_cal"})

# Métriques après calibration
cmp2 = haz_obs[["t","ER_marginal"]].rename(columns={"ER_marginal":"obs"}).merge(
       curve_cal[["t","ER_marginal_cal"]].rename(columns={"ER_marginal_cal":"pred"}), on="t")
mae_after  = (cmp2["obs"] - cmp2["pred"]).abs().mean()
rmse_after = np.sqrt(((cmp2["obs"] - cmp2["pred"])**2).mean())

print(f"Après calibration — MAE : {mae_after:.6f}  |  RMSE : {rmse_after:.6f}")
print(f"Gain MAE : {mae_before - mae_after:.6f}")
print()
print(curve_cal[["t","ER_marginal_cal","ER_cumulee_cal"]].head(10).to_string(index=False))

### Visualisation des courbes prédites calibrées vs observées

In [ ]:
cmp_plot = haz_obs.merge(curve_cal[["t","ER_marginal_cal","ER_cumulee_cal"]], on="t", how="inner")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cmp_plot["t"], cmp_plot["ER_marginal"],     label="Observée (test)",   linewidth=1.5)
axes[0].plot(cmp_plot["t"], cmp_plot["ER_marginal_cal"], label="Prédite calibrée",  linewidth=1.5, linestyle="--")
axes[0].set_xlabel("t (mois)")
axes[0].set_ylabel("ER marginale")
axes[0].set_title("Taux d'extinction marginale")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(cmp_plot["t"], cmp_plot["ER_cumulee"],     label="Observée (test)",  linewidth=1.5)
axes[1].plot(cmp_plot["t"], cmp_plot["ER_cumulee_cal"], label="Prédite calibrée", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("t (mois)")
axes[1].set_ylabel("ER cumulée")
axes[1].set_title("Taux d'extinction cumulée")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Comparaison ER prédite calibrée vs observée — jeu de test", fontsize=12)
plt.tight_layout()
plt.show()

## Etape 5 : Backtesting temporel sur fenêtres disjointes

### Protocole

On réalise un backtesting sur **3 fenêtres temporelles disjointes** (*non-overlapping walk-forward validation*). Les cuts sont définis aux percentiles 50%, 65% et 80% de la date d'origination. Les jeux de test sont construits de façon à ne pas se chevaucher :

| Fold | Train | Test |
|------|-------|------|
| 1 | $[0, p_{50}]$ | $(p_{50}, p_{65}]$ |
| 2 | $[0, p_{65}]$ | $(p_{65}, p_{80}]$ |
| 3 | $[0, p_{80}]$ | $(p_{80}, +\infty)$ |

Cette structure est plus conservative qu'un backtesting à test sets chevauchants, et permet d'évaluer la performance sur des périodes réellement distinctes.

A chaque fold, le modèle est ré-estimé sur le train set (standardisation incluse), puis calibré sur le test set correspondant.

In [ ]:
# Quantiles définissant les bornes des fenêtres disjointes
q_cuts = [0.50, 0.65, 0.80]
cut_dates = base[DATE_COL].quantile(q_cuts).values

# Définition des intervalles de test disjoints
# fold k : test = (cut_dates[k-1], cut_dates[k]] pour k = 1, 2
#           test = (cut_dates[2], +inf)            pour k = 3
fold_definitions = [
    (None,          cut_dates[0], cut_dates[0], cut_dates[1]),   # fold 1
    (None,          cut_dates[1], cut_dates[1], cut_dates[2]),   # fold 2
    (None,          cut_dates[2], cut_dates[2], None),           # fold 3
]

results = []

for k, (_, train_end, test_start, test_end) in enumerate(fold_definitions, 1):

    # Sélection des contrats par fenêtre
    train_mask = base[DATE_COL] <= train_end
    if test_end is not None:
        test_mask = (base[DATE_COL] > test_start) & (base[DATE_COL] <= test_end)
    else:
        test_mask = base[DATE_COL] > test_start

    train_ids_k = set(base.loc[train_mask, ID_COL])
    test_ids_k  = set(base.loc[test_mask,  ID_COL])

    # Exclusion des contrats test qui seraient aussi dans le train
    test_ids_k = test_ids_k - train_ids_k

    if len(test_ids_k) == 0:
        print(f"Fold {k} : jeu de test vide, fold ignoré.")
        continue

    train_k = panel_cov[panel_cov[ID_COL].isin(train_ids_k)].dropna(subset=needed).copy()
    test_k  = panel_cov[panel_cov[ID_COL].isin(test_ids_k)].dropna(subset=needed).copy()

    # Standardisation sur le train du fold
    tx_m, tx_s    = train_k["Tx"].mean(),      train_k["Tx"].std(ddof=0)
    age_m, age_s  = train_k["AGE_CLI"].mean(),  train_k["AGE_CLI"].std(ddof=0)
    train_k["Tx_z"]      = (train_k["Tx"]      - tx_m)  / (tx_s  if tx_s  > 0 else 1)
    test_k["Tx_z"]       = (test_k["Tx"]       - tx_m)  / (tx_s  if tx_s  > 0 else 1)
    train_k["AGE_CLI_z"] = (train_k["AGE_CLI"] - age_m) / (age_s if age_s > 0 else 1)
    test_k["AGE_CLI_z"]  = (test_k["AGE_CLI"]  - age_m) / (age_s if age_s > 0 else 1)

    if has_mrevtot:
        rev_m, rev_s = train_k["MREVTOT"].mean(), train_k["MREVTOT"].std(ddof=0)
        train_k["MREVTOT_z"] = (train_k["MREVTOT"] - rev_m) / (rev_s if rev_s > 0 else 1)
        test_k["MREVTOT_z"]  = (test_k["MREVTOT"]  - rev_m) / (rev_s if rev_s > 0 else 1)

    # Estimation
    y_tr_k, X_tr_k = patsy.dmatrices(formula, data=train_k, return_type="dataframe")
    res_k = sm.GLM(y_tr_k, X_tr_k, family=sm.families.Binomial()).fit()

    _, X_te_k = patsy.dmatrices(formula, data=test_k.assign(y=0), return_type="dataframe")
    p0_k = res_k.predict(X_te_k).clip(1e-9, 1 - 1e-9)

    # Cible de calibration : calculée sur les contrats test uniquement
    test_contracts_k = df_base[df_base[ID_COL].isin(test_ids_k)]
    target_k = test_contracts_k["EVENT"].mean()

    # Calibration
    lo_k, hi_k = -10.0, 10.0
    for _ in range(30):
        mid_k = (lo_k + hi_k) / 2
        if ER_final_with_delta(mid_k, test_k[ID_COL].values, test_k["t"].values, p0_k) < target_k:
            lo_k = mid_k
        else:
            hi_k = mid_k
    delta_k = (lo_k + hi_k) / 2

    p_cal_k   = expit(logit(p0_k) + delta_k)
    curve_k   = compute_portfolio_curve(test_k[ID_COL].values, test_k["t"].values, p_cal_k)

    # Courbe observée
    haz_obs_k = test_k.groupby("t", as_index=False).agg(N=("y","size"), E=("y","sum"))
    haz_obs_k["h"]          = haz_obs_k["E"] / haz_obs_k["N"]
    haz_obs_k["S_t"]        = (1 - haz_obs_k["h"]).cumprod()
    haz_obs_k["ER_marginal"]= haz_obs_k["S_t"].shift(1).fillna(1.0) - haz_obs_k["S_t"]

    cmp_k = haz_obs_k[["t","ER_marginal"]].rename(columns={"ER_marginal":"obs"}).merge(
            curve_k[["t","ER_marginal"]].rename(columns={"ER_marginal":"pred"}), on="t")

    mae_k  = (cmp_k["obs"] - cmp_k["pred"]).abs().mean()
    rmse_k = np.sqrt(((cmp_k["obs"] - cmp_k["pred"])**2).mean())

    test_start_date = pd.to_datetime(test_start).date()
    test_end_date   = pd.to_datetime(test_end).date() if test_end is not None else "fin"

    results.append({
        "fold": k,
        "train_end":     pd.to_datetime(train_end).date(),
        "test_window":   f"{test_start_date} -> {test_end_date}",
        "n_contrats_test": len(test_ids_k),
        "target_EVENT":  round(target_k, 6),
        "delta_star":    round(delta_k, 4),
        "MAE":           round(mae_k, 6),
        "RMSE":          round(rmse_k, 6),
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

### Interprétation du backtesting

Les fenêtres de test sont disjointes, ce qui garantit que chaque contrat n'est évalué qu'une seule fois. Les métriques (MAE, RMSE sur l'ER marginale) et le `delta_star` de calibration permettent d'évaluer :

- **La stabilité du modèle** : des métriques homogènes entre les folds indiquent une bonne robustesse temporelle.
- **L'ampleur du biais résiduel** : un `delta_star` stable et proche de 0 indique que le modèle est bien calibré sans correction importante. Un `delta_star` élevé ou variable suggère un problème structurel (effets millésime, dérive du portefeuille).
- **La dégradation sur les données récentes** : une MAE plus élevée sur le fold 3 (données les plus récentes) peut signaler un concept drift ou des changements de comportement des emprunteurs non capturés par les covariables actuelles.

## Export de la table finale

La table `er_final` contient les courbes d'extinction calibrées (ER marginale et cumulée) par mois $t \in [0, 60]$, estimées sur le split principal (train 80% / test 20%).

In [ ]:
er_final = curve_cal[["t", "ER_marginal_cal", "ER_cumulee_cal"]].copy()
print(er_final.to_string(index=False))

# Décommentez pour exporter :
# er_final.to_csv("ER_pred_calibree.csv", index=False)